# Práctica 1er Parcial — SQL y NoSQL

**Estudiante:** Brailin Leonardo Méndez Guillermo  
**Objetivo:** Ejecutar operaciones CRUD en SQLite y MongoDB, además de trabajar con una colección de productos.


## Parte A — SQLite
La base se crea en memoria para que el cuaderno sea reproducible.

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()
cursor.execute(
    "CREATE TABLE users (id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT NOT NULL, age INTEGER NOT NULL)"
)
cursor.executemany(
    "INSERT INTO users (name, age) VALUES (?, ?)",
    [("Alice", 25), ("Bob", 30), ("Charlie", 35)],
)
conn.commit()
pd.read_sql_query("SELECT * FROM users", conn)


In [ ]:
# 1. Usuarios mayores de 28 años
pd.read_sql_query("SELECT * FROM users WHERE age > 28 ORDER BY age", conn)


In [ ]:
# 2. INSERT
cursor.execute("INSERT INTO users (name, age) VALUES (?, ?)", ("David", 27))
# 3. UPDATE
cursor.execute("UPDATE users SET age = ? WHERE name = ?", (26, "Alice"))
# 4. DELETE
cursor.execute("DELETE FROM users WHERE name = ?", ("Bob",))
conn.commit()
pd.read_sql_query("SELECT * FROM users ORDER BY id", conn)


## Parte B — MongoDB Atlas
Guarda `MONGO_URI` como secreto de Colab. La URI nunca se imprime.

In [ ]:
!pip -q install pymongo[srv]
import os
from getpass import getpass
from pymongo import MongoClient

MONGO_URI = os.environ.get("MONGO_URI") or getpass("MONGO_URI de Atlas: " )
client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=10000)
client.admin.command("ping")
print("Conexión segura a MongoDB Atlas verificada")


In [ ]:
db = client["curso_nosql"]
usuarios = db["usuarios"]
usuarios.delete_many({"practica": "itla-2026-c3"})
usuarios.insert_many([
    {"name": "Alice", "age": 25, "practica": "itla-2026-c3"},
    {"name": "Bob", "age": 30, "practica": "itla-2026-c3"},
    {"name": "Charlie", "age": 35, "practica": "itla-2026-c3"},
])
list(usuarios.find({"practica": "itla-2026-c3"}, {"_id": 0}))


In [ ]:
# CRUD solicitado
mayores_28 = list(usuarios.find(
    {"age": {"$gt": 28}, "practica": "itla-2026-c3"},
    {"_id": 0}
))
usuarios.insert_one({"name": "David", "age": 27, "practica": "itla-2026-c3"})
usuarios.update_one({"name": "Alice", "practica": "itla-2026-c3"}, {"$set": {"age": 26}})
usuarios.delete_one({"name": "Bob", "practica": "itla-2026-c3"})
print("Mayores de 28:", mayores_28)
print("Estado final:", list(usuarios.find({"practica": "itla-2026-c3"}, {"_id": 0})))


In [ ]:
# Ejercicio adicional: productos
productos = db["productos"]
productos.delete_many({"practica": "itla-2026-c3"})
productos.insert_many([
    {"nombre": "Laptop", "precio": 45000, "stock": 10, "practica": "itla-2026-c3"},
    {"nombre": "Mouse", "precio": 800, "stock": 50, "practica": "itla-2026-c3"},
    {"nombre": "Teclado", "precio": 1500, "stock": 25, "practica": "itla-2026-c3"},
])
print("Todos:", list(productos.find({"practica": "itla-2026-c3"}, {"_id": 0})))
print("Precio > 1000:", list(productos.find({"precio": {"$gt": 1000}, "practica": "itla-2026-c3"}, {"_id": 0})))
productos.update_one({"nombre": "Mouse", "practica": "itla-2026-c3"}, {"$set": {"stock": 45}})
productos.delete_one({"nombre": "Teclado", "practica": "itla-2026-c3"})
print("Resultado final:", list(productos.find({"practica": "itla-2026-c3"}, {"_id": 0})))
